In [ ]:
!pwd

/content/drive/MyDrive/LUNA16/Lung-nodule-detection-LUNA-16


In [ ]:
# !mv ../annotations.csv dataset

In [ ]:
!ls

data  data_prep  dataset  LICENSE  Plots  README.md  train_codes


In [ ]:
import os
os.chdir('/content/drive/MyDrive/LUNA16/Lung-nodule-detection-LUNA-16')

In [ ]:
# Run this once
# !git clone https://github.com/Rakshith2597/Lung-nodule-detection-LUNA-16.git

## Inspect the dataset

In [ ]:
!pip install SimpleITK
%matplotlib inline
import SimpleITK as sitk
import numpy as np
import csv
from glob import glob
import pandas as pd
import matplotlib.pyplot as plt
import os
from tqdm import tqdm_notebook as tq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 MB 19.4 MB/s eta 0:00:00


In [ ]:
def get_filename(case):
    global file_list
    for f in file_list:
        if case in f:
            return(f)

In [ ]:
def make_mask(center,diam,z,width,height,spacing,origin):
    mask = np.zeros([height,width]) # 0's everywhere except nodule swapping x,y to match img
    #convert to nodule space from world coordinates

    # Defining the voxel range in which the nodule falls
    v_center = (center-origin)/spacing
    v_diam = int((diam+5)/spacing[0])
    v_xmin = np.max([0,int(v_center[0]-v_diam)])
    v_xmax = np.min([width-1,int(v_center[0]+v_diam)])
    v_ymin = np.max([0,int(v_center[1]-v_diam)])
    v_ymax = np.min([height-1,int(v_center[1]+v_diam)])

    v_xrange = range(v_xmin,v_xmax+1)
    v_yrange = range(v_ymin,v_ymax+1)

    # Convert back to world coordinates for distance calculation
    x_data = [x*spacing[0]+origin[0] for x in range(width)]
    y_data = [x*spacing[1]+origin[1] for x in range(height)]
    for v_x in v_xrange:
        for v_y in v_yrange:
            p_x = spacing[0]*v_x + origin[0]
            p_y = spacing[1]*v_y + origin[1]
            if np.linalg.norm(center-np.array([p_x,p_y,z]))<=diam:
                mask[int((p_y-origin[1])/spacing[1]),int((p_x-origin[0])/spacing[0])] = 1.0
    return(mask)

In [ ]:
def matrix2int16(matrix):
    '''
matrix must be a numpy array NXN
Returns uint16 version
    '''
    m_min= np.min(matrix)
    m_max= np.max(matrix)
    matrix = matrix-m_min
    return(np.array(np.rint( (matrix-m_min)/float(m_max-m_min) * 65535.0),dtype=np.uint16))

In [ ]:
# luna_path="./dataset/"
# tr_output_img_path="./data/train/images/"
# tr_output_mask_path="./data/train/labels/"

# if not os.path.isdir(tr_output_img_path):
#     os.makedirs(tr_output_img_path)
# if not os.path.isdir(tr_output_mask_path):
#     os.makedirs(tr_output_mask_path)

# v_output_img_path="./data/val/images/"
# v_output_mask_path="./data/val/labels/"

# if not os.path.isdir(v_output_img_path):
#     os.makedirs(v_output_img_path)
# if not os.path.isdir(v_output_mask_path):
#     os.makedirs(v_output_mask_path)

# ts_output_img_path="./data/test/images/"
# ts_output_mask_path="./data/test/labels/"

# if not os.path.isdir(ts_output_img_path):
#     os.makedirs(ts_output_img_path)
# if not os.path.isdir(ts_output_mask_path):
#     os.makedirs(ts_output_mask_path)

In [ ]:
#!cat ./dataset/annotations.csv

In [ ]:
import os
os.chdir('/content/drive/MyDrive/LUNA16/Lung-nodule-detection-LUNA-16')

In [ ]:
!pip install SimpleITK
%matplotlib inline
import SimpleITK as sitk
import numpy as np
import csv
from glob import glob
import pandas as pd
import matplotlib.pyplot as plt
import os
from tqdm import tqdm_notebook as tq

1095


In [ ]:
loadPath = './data/train/merge_images/'
trFiles = os.listdir(loadPath)
print(len(trFiles))

# initialize accumulators
channel_sum = np.zeros(3, dtype=np.float64)
channel_sum_sq = np.zeros(3, dtype=np.float64)
num_pixels = 0  # total pixels per channel

for f in tq(trFiles):
    img = np.load(loadPath + "/" + f)   # shape: (3, 512, 512)

    # sum over H and W only
    channel_sum     += img.sum(axis=(1, 2))
    channel_sum_sq  += (img ** 2).sum(axis=(1, 2))

    num_pixels += img.shape[1] * img.shape[2]  # 512 * 512

# mean and std per channel
mean = channel_sum / num_pixels
std = np.sqrt(channel_sum_sq / num_pixels - mean**2)

print("Mean:", mean)
print("Std:", std)


/tmp/ipython-input-2246205919.py:6: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for f in tq(trFiles):


  0%|          | 0/1095 [00:00<?, ?it/s]

Mean: [-4.64620118e+02 -8.74013723e-04 -7.96379932e-04]
Std: [4.44225775e+02 3.78290289e-03 3.83985386e-03]


In [ ]:
# print('Mean:',totalData.mean(),'Std:',totalData.std())
# print('Min:',totalData.min(),'Max:',totalData.max())

## Trainer

Always run from this step !!

### Dataloader

### Lovaz Losses

In [ ]:
"""
Lovasz-Softmax and Jaccard hinge loss in PyTorch
Maxim Berman 2018 ESAT-PSI KU Leuven (MIT License)
"""

from __future__ import print_function, division

import torch
from torch.autograd import Variable
import torch.nn.functional as F
import numpy as np
try:
    from itertools import  ifilterfalse
except ImportError: # py3k
    from itertools import  filterfalse as ifilterfalse


def lovasz_grad(gt_sorted):
    """
    Computes gradient of the Lovasz extension w.r.t sorted errors
    See Alg. 1 in paper
    """
    p = len(gt_sorted)
    gts = gt_sorted.sum()
    intersection = gts - gt_sorted.float().cumsum(0)
    union = gts + (1 - gt_sorted).float().cumsum(0)
    jaccard = 1. - intersection / union
    if p > 1: # cover 1-pixel case
        jaccard[1:p] = jaccard[1:p] - jaccard[0:-1]
    return jaccard


def iou_binary(preds, labels, EMPTY=1., ignore=None, per_image=True):
    """
    IoU for foreground class
    binary: 1 foreground, 0 background
    """
    if not per_image:
        preds, labels = (preds,), (labels,)
    ious = []
    for pred, label in zip(preds, labels):
        intersection = ((label == 1) & (pred == 1)).sum()
        union = ((label == 1) | ((pred == 1) & (label != ignore))).sum()
        if not union:
            iou = EMPTY
        else:
            iou = float(intersection) / float(union)
        ious.append(iou)
    iou = mean(ious)    # mean accross images if per_image
    return 100 * iou


def iou(preds, labels, C, EMPTY=1., ignore=None, per_image=False):
    """
    Array of IoU for each (non ignored) class
    """
    if not per_image:
        preds, labels = (preds,), (labels,)
    ious = []
    for pred, label in zip(preds, labels):
        iou = []
        for i in range(C):
            if i != ignore: # The ignored label is sometimes among predicted classes (ENet - CityScapes)
                intersection = ((label == i) & (pred == i)).sum()
                union = ((label == i) | ((pred == i) & (label != ignore))).sum()
                if not union:
                    iou.append(EMPTY)
                else:
                    iou.append(float(intersection) / float(union))
        ious.append(iou)
    ious = [mean(iou) for iou in zip(*ious)] # mean accross images if per_image
    return 100 * np.array(ious)


# --------------------------- BINARY LOSSES ---------------------------


def lovasz_hinge(logits, labels, per_image=True, ignore=None):
    """
    Binary Lovasz hinge loss
      logits: [B, H, W] Variable, logits at each pixel (between -\infty and +\infty)
      labels: [B, H, W] Tensor, binary ground truth masks (0 or 1)
      per_image: compute the loss per image instead of per batch
      ignore: void class id
    """
    if per_image:
        loss = mean(lovasz_hinge_flat(*flatten_binary_scores(log.unsqueeze(0), lab.unsqueeze(0), ignore))
                          for log, lab in zip(logits, labels))
    else:
        loss = lovasz_hinge_flat(*flatten_binary_scores(logits, labels, ignore))
    return loss


def lovasz_hinge_flat(logits, labels):
    """
    Binary Lovasz hinge loss
      logits: [P] Variable, logits at each prediction (between -\infty and +\infty)
      labels: [P] Tensor, binary ground truth labels (0 or 1)
      ignore: label to ignore
    """
    if len(labels) == 0:
        # only void pixels, the gradients should be 0
        return logits.sum() * 0.
    signs = 2. * labels.float() - 1.
    errors = (1. - logits * Variable(signs))
    errors_sorted, perm = torch.sort(errors, dim=0, descending=True)
    perm = perm.data
    gt_sorted = labels[perm]
    grad = lovasz_grad(gt_sorted)
    loss = torch.dot(F.relu(errors_sorted), Variable(grad))
    return loss


def flatten_binary_scores(scores, labels, ignore=None):
    """
    Flattens predictions in the batch (binary case)
    Remove labels equal to 'ignore'
    """
    scores = scores.view(-1)
    labels = labels.view(-1)
    if ignore is None:
        return scores, labels
    valid = (labels != ignore)
    vscores = scores[valid]
    vlabels = labels[valid]
    return vscores, vlabels


class StableBCELoss(torch.nn.modules.Module):
    def __init__(self):
         super(StableBCELoss, self).__init__()
    def forward(self, input, target):
         neg_abs = - input.abs()
         loss = input.clamp(min=0) - input * target + (1 + neg_abs.exp()).log()
         return loss.mean()


def binary_xloss(logits, labels, ignore=None):
    """
    Binary Cross entropy loss
      logits: [B, H, W] Variable, logits at each pixel (between -\infty and +\infty)
      labels: [B, H, W] Tensor, binary ground truth masks (0 or 1)
      ignore: void class id
    """
    logits, labels = flatten_binary_scores(logits, labels, ignore)
    loss = StableBCELoss()(logits, Variable(labels.float()))
    return loss


# --------------------------- MULTICLASS LOSSES ---------------------------


def lovasz_softmax(probas, labels, classes='present', per_image=False, ignore=None):
    """
    Multi-class Lovasz-Softmax loss
      probas: [B, C, H, W] Variable, class probabilities at each prediction (between 0 and 1).
              Interpreted as binary (sigmoid) output with outputs of size [B, H, W].
      labels: [B, H, W] Tensor, ground truth labels (between 0 and C - 1)
      classes: 'all' for all, 'present' for classes present in labels, or a list of classes to average.
      per_image: compute the loss per image instead of per batch
      ignore: void class labels
    """
    if per_image:
        loss = mean(lovasz_softmax_flat(*flatten_probas(prob.unsqueeze(0), lab.unsqueeze(0), ignore), classes=classes)
                          for prob, lab in zip(probas, labels))
    else:
        loss = lovasz_softmax_flat(*flatten_probas(probas, labels, ignore), classes=classes)
    return loss

def lovasz_softmax_flat(probas, labels, classes='present'):
    if probas.numel() == 0:
        return probas * 0.
    C = probas.size(1)
    losses = []
    class_to_sum = list(range(C)) if classes in ['all', 'present'] else classes
    for c in class_to_sum:
        fg = (labels == c).float()
        if (classes == 'present' and fg.sum() == 0):
            continue
        if C == 1:
            if len(classes) > 1:
                raise ValueError('Sigmoid output possible only with 1 class')
            class_pred = probas[:, 0]
        else:
            class_pred = probas[:, c]

        errors = (fg - class_pred).abs()
        errors_sorted, perm = torch.sort(errors, 0, descending=True)
        perm = perm.data
        fg_sorted = fg[perm]

        losses.append(torch.dot(errors_sorted, lovasz_grad(fg_sorted)))

    return torch.mean(torch.stack(losses))

# def lovasz_softmax_flat(probas, labels, classes='present'):
#     """
#     Multi-class Lovasz-Softmax loss
#       probas: [P, C] Variable, class probabilities at each prediction (between 0 and 1)
#       labels: [P] Tensor, ground truth labels (between 0 and C - 1)
#       classes: 'all' for all, 'present' for classes present in labels, or a list of classes to average.
#     """
#     if probas.numel() == 0:
#         # only void pixels, the gradients should be 0
#         return probas * 0.
#     C = probas.size(1)
#     losses = []
#     class_to_sum = list(range(C)) if classes in ['all', 'present'] else classes
#     for c in class_to_sum:
#         fg = (labels == c).float() # foreground for class c
#         if (classes is 'present' and fg.sum() == 0):
#             continue
#         if C == 1:
#             if len(classes) > 1:
#                 raise ValueError('Sigmoid output possible only with 1 class')
#             class_pred = probas[:, 0]
#         else:
#             class_pred = probas[:, c]
#         errors = (Variable(fg) - class_pred).abs()
#         errors_sorted, perm = torch.sort(errors, 0, descending=True)
#         perm = perm.data
#         fg_sorted = fg[perm]
#         losses.append(torch.dot(errors_sorted, Variable(lovasz_grad(fg_sorted))))
#     return mean(losses)


def flatten_probas(probas, labels, ignore=None):
    """
    Flattens predictions in the batch
    """
    if probas.dim() == 3:
        # assumes output of a sigmoid layer
        B, H, W = probas.size()
        probas = probas.view(B, 1, H, W)
    B, C, H, W = probas.size()
    probas = probas.permute(0, 2, 3, 1).contiguous().view(-1, C)  # B * H * W, C = P, C
    labels = labels.view(-1)
    if ignore is None:
        return probas, labels
    valid = (labels != ignore)
    vprobas = probas[valid.nonzero().squeeze()]
    vlabels = labels[valid]
    return vprobas, vlabels

def xloss(logits, labels, ignore=None):
    """
    Cross entropy loss
    """
    return F.cross_entropy(logits, Variable(labels), ignore_index=255)


# --------------------------- HELPER FUNCTIONS ---------------------------
def isnan(x):
    return x != x


def mean(l, ignore_nan=False, empty=0):
    """
    nanmean compatible with generators.
    """
    l = iter(l)
    if ignore_nan:
        l = ifilterfalse(isnan, l)
    try:
        n = 1
        acc = next(l)
    except StopIteration:
        if empty == 'raise':
            raise ValueError('Empty mean')
        return empty
    for n, v in enumerate(l, 2):
        acc += v
    if n == 1:
        return acc
    return acc / n

<>:81: SyntaxWarning: invalid escape sequence '\i'
<>:97: SyntaxWarning: invalid escape sequence '\i'
<>:141: SyntaxWarning: invalid escape sequence '\i'
<>:81: SyntaxWarning: invalid escape sequence '\i'
<>:97: SyntaxWarning: invalid escape sequence '\i'
<>:141: SyntaxWarning: invalid escape sequence '\i'
/tmp/ipython-input-2326729339.py:81: SyntaxWarning: invalid escape sequence '\i'
  logits: [B, H, W] Variable, logits at each pixel (between -\infty and +\infty)
/tmp/ipython-input-2326729339.py:97: SyntaxWarning: invalid escape sequence '\i'
  logits: [P] Variable, logits at each prediction (between -\infty and +\infty)
/tmp/ipython-input-2326729339.py:141: SyntaxWarning: invalid escape sequence '\i'
  logits: [B, H, W] Variable, logits at each pixel (between -\infty and +\infty)


### Train

In [ ]:
!pip install segmentation-models-pytorch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 5.2 MB/s eta 0:00:00


In [ ]:
#Code written by Rakshith Sathish
#The work is made public with MIT License

import os
import collections
import torch
import numpy as np
import scipy.misc as m
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms


from torch.utils import data


# Corrected lunaLoader (NO PIL for images)

class lunaLoader(data.Dataset):
    def __init__(self, split="train", is_transform=True, img_size=512, mean=None, std=None):

        self.split = split
        self.path = "./data/" + split
        self.is_transform = is_transform
        self.img_size = img_size

        self.files = os.listdir(self.path + '/merge_images/')

        if mean is None or std is None:
            raise ValueError("You must pass mean and std (per channel)")

        # Transforms for images AFTER converted to tensor
        self.img_tf = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.Normalize(mean, std)
        ])

        # Label transform
        self.label_tf = transforms.Resize((img_size, img_size), interpolation=0)


    def __len__(self):
        return len(self.files)


    def __getitem__(self, index):
        fname = self.files[index]

        # -----------------------------
        # Load IMAGE (numpy → tensor)
        # -----------------------------
        img_np = np.load(self.path + '/merge_images/' + fname)   # shape (3,512,512)

        if img_np.shape[0] == 3:     # (C,H,W) → (H,W,C)
            img_np = np.transpose(img_np, (1, 2, 0))

        img = torch.from_numpy(img_np).float()        # (H,W,C)
        img = img.permute(2, 0, 1)                    # (C,H,W)

        # -----------------------------
        # Load LABEL
        # -----------------------------
        im_id = fname.split('_')[1]
        lbl_np = np.load(self.path + '/labels/masks_' + im_id)

        label = torch.from_numpy(lbl_np).long().unsqueeze(0)  # (1,H,W)

        # -----------------------------
        # Apply transforms
        # -----------------------------
        if self.is_transform:
            img = self.img_tf(img)
            label = self.label_tf(label)

        return img, label.squeeze(0)


In [ ]:
loadPath = './data/train/merge_images/'
trFiles = os.listdir(loadPath)
print(len(trFiles))

# initialize accumulators
channel_sum = np.zeros(3, dtype=np.float64)
channel_sum_sq = np.zeros(3, dtype=np.float64)
num_pixels = 0  # total pixels per channel

for f in tq(trFiles):
    img = np.load(loadPath + "/" + f)   # shape: (3, 512, 512)

    # sum over H and W only
    channel_sum     += img.sum(axis=(1, 2))
    channel_sum_sq  += (img ** 2).sum(axis=(1, 2))

    num_pixels += img.shape[1] * img.shape[2]  # 512 * 512

# mean and std per channel
train_mean = channel_sum / num_pixels
train_std = np.sqrt(channel_sum_sq / num_pixels - train_mean**2)

print("Mean:", train_mean)
print("Std:", train_std)


1095


/tmp/ipython-input-389433994.py:10: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for f in tq(trFiles):


  0%|          | 0/1095 [00:00<?, ?it/s]

Mean: [-4.64620118e+02 -8.74013723e-04 -7.96379932e-04]
Std: [4.44225775e+02 3.78290289e-03 3.83985386e-03]


In [ ]:

#Code written by Rakshith Sathish
#The work is made public with MIT License

import segmentation_models_pytorch as smp

import numpy as np
import torch
import torch.nn as nn
from torch import optim
import tqdm
import time
from torch.utils import data
import os
import torch.nn
import torch.nn.functional as F
from torch.autograd import  Variable
import matplotlib.pyplot as plt
plt.switch_backend('agg')
from sklearn.metrics import confusion_matrix
# from SUMNet_bn import SUMNet
# from LUNA_loader import lunaLoader
# import lovasz_losses as L

def dice_coefficient(pred, target):
	predC = torch.argmax(F.softmax(pred,dim=1),dim=1)
	c = confusion_matrix(target.view(-1).cpu().numpy(), predC.view(-1).cpu().numpy(),labels=[0,1])
	TP = np.diag(c)
	FP = c.sum(axis=0) - np.diag(c)
	FN = c.sum(axis=1) - np.diag(c)
	TN = c.sum() - (FP + FN + TP)
	return (TP,FP,FN)



savePath = 'Results/UnetCeNN/Adam_1e-4_ep100_CE+Lov/'
if not os.path.isdir(savePath):
	os.makedirs(savePath)


trainDset = lunaLoader(is_transform=True, split='train',img_size=256, mean=train_mean, std=train_std)
valDset = lunaLoader(is_transform=True, split='val',img_size=256, mean=train_mean, std=train_std)

trainDataLoader = data.DataLoader(trainDset,batch_size=16,shuffle=True,num_workers=4,pin_memory=True)
validDataLoader = data.DataLoader(valDset,batch_size=16,shuffle=False,num_workers=4,pin_memory=True)

n_classes = 2
net_in_channels = 3
# net = SUMNet(in_ch=1,out_ch=n_classes)

net = smp.Unet(
    encoder_name="resnet34",        # choose encoder, e.g. mobilenet_v2 or efficientnet-b7
    encoder_weights="imagenet",     # use `imagenet` pre-trained weights for encoder initialization
    in_channels=net_in_channels,                  # model input channels (1 for gray-scale images, 3 for RGB, etc.)
    classes=n_classes,                      # model output channels (number of classes in your dataset)
)

use_gpu = torch.cuda.is_available()
if use_gpu:
	net = net.cuda()

optimizerS = optim.Adam(net.parameters(), lr = 1e-4, weight_decay = 1e-5)
criterionS = nn.CrossEntropyLoss()

epochs = 100
trainLoss = []
validLoss = []
trainDiceCoeff = []
validDiceCoeff = []
start = time.time()

bestValidDice = 0.0

for epoch in range(epochs):
	epochStart = time.time()
	trainRunningLoss = 0
	validRunningLoss = 0
	trainBatches = 0
	validBatches = 0

	train_tp = np.zeros(n_classes)
	train_fp = np.zeros(n_classes)
	train_fn = np.zeros(n_classes)

	val_tp = np.zeros(n_classes)
	val_fp = np.zeros(n_classes)
	val_fn = np.zeros(n_classes)


	net.train(True)
	for data1 in tqdm.tqdm(trainDataLoader):
		imgs, mask = data1
		# print(imgs.shape)
		if use_gpu:
			inputs = imgs.cuda()
			labels = mask.cuda()


		cpmap = net(Variable(inputs))
		cpmapD = F.softmax(cpmap,dim=1)

		LGce = criterionS(cpmap,labels.long())
		L_lov = lovasz_softmax(F.softmax(cpmap,dim=1),labels)
		LGseg = LGce+L_lov

		optimizerS.zero_grad()

		LGseg.backward()

		optimizerS.step()

		trainRunningLoss += LGseg.item()

		train_cf = dice_coefficient(cpmapD,labels)
		train_tp += train_cf[0]
		train_fp += train_cf[1]
		train_fn += train_cf[2]
		trainBatches += 1
		# break


	train_dice = (2*train_tp)/(2*train_tp + train_fp + train_fn )
	trainLoss.append(trainRunningLoss/trainBatches)
	trainDiceCoeff.append(train_dice)

	print("\n{}][{}]| LGseg: {:.4f} | "
		.format(epoch,epochs,LGseg.item()))

	with torch.no_grad():
		for data1 in tqdm.tqdm(validDataLoader):
			imgs, mask = data1
			if use_gpu:
				inputs = imgs.cuda()
				labels = mask.cuda()


			cpmap = net(Variable(inputs))
			cpmapD = F.softmax(cpmap.data,dim=1)

			val_cf = dice_coefficient(cpmapD,labels)
			val_tp += val_cf[0]
			val_fp += val_cf[1]
			val_fn += val_cf[2]
			validRunningLoss += LGseg.item()
			validBatches += 1
			# break


		val_dice = (2*val_tp)/(2*val_tp + val_fp + val_fn )
		validLoss.append(validRunningLoss/validBatches)
		validDiceCoeff.append(val_dice)
	# scheduler.step(validRunningLoss/validBatches)
	if (val_dice[1] > bestValidDice):
		print(f"Found a better version, old dice loss: {bestValidDice} -> New dice loss: {val_dice[1]}. Saving checkpoint...")
		bestValidDice = val_dice[1]
		torch.save(net.state_dict(), savePath+'sumnet_best.pt')


	plt.figure()
	plt.plot(range(len(trainLoss)),trainLoss,'-r',label='Train')
	plt.plot(range(len(validLoss)),validLoss,'-g',label='Valid')
	if epoch==0:
		plt.legend()
	plt.savefig(savePath+'LossPlot.png')
	plt.close()
	epochEnd = time.time()-epochStart
	print('Epoch: {:.0f}/{:.0f} | Train Loss: {:.3f} | Valid Loss: {:.3f}'\
		  .format(epoch+1, epochs, trainRunningLoss/trainBatches, validRunningLoss/validBatches))
	print('\nDice | Train  | BG {:.3f} | Nodule {:.3f} |\n Valid | BG: {:.3f} | Nodule {:.3f} |'
		  .format(train_dice[0],train_dice[1], val_dice[0], val_dice[1]))

	print('\nTime: {:.0f}m {:.0f}s'.format(epochEnd//60,epochEnd%60))
	trainLoss_np = np.array(trainLoss)
	validLoss_np = np.array(validLoss)
	trainDiceCoeff_np = np.array(trainDiceCoeff)
	validDiceCoeff_np = np.array(validDiceCoeff)

	print('Saving losses')

	torch.save(trainLoss_np, savePath+'trainLoss.pt')
	torch.save(validLoss_np, savePath+'validLoss.pt')
	torch.save(trainDiceCoeff_np, savePath+'trainDice.pt')
	torch.save(validDiceCoeff_np, savePath+'validDice.pt')
	# break


end = time.time()-start
print('Training completed in {:.0f}m {:.0f}s'.format(end//60,end%60))
plt.figure()
plt.plot(range(len(trainLoss)),trainLoss,'-r')
plt.plot(range(len(validLoss)),validLoss,'-g')
plt.title('Loss plot')
plt.savefig(savePath+'trainLossFinal.png')
plt.close()

trainDiceCoeff_bg = [x[0] for x in trainDiceCoeff]
trainDiceCoeff_nodule = [x[1] for x in trainDiceCoeff]
plt.figure()
plt.plot(range(len(trainDiceCoeff_bg)),trainDiceCoeff_bg,'-r',label='BG')
plt.plot(range(len(trainDiceCoeff_nodule)),trainDiceCoeff_nodule,'-g',label='Nodule')
plt.legend()
plt.title('Dice coefficient: Train')
plt.savefig(savePath+'trainDice.png')
plt.close()

validDiceCoeff_bg = [x[0] for x in validDiceCoeff]
validDiceCoeff_nodule = [x[1] for x in validDiceCoeff]
plt.figure()
plt.plot(range(len(validDiceCoeff_bg)),validDiceCoeff_bg,'-r',label='BG')
plt.plot(range(len(validDiceCoeff_nodule)),validDiceCoeff_nodule,'-g',label='Nodule')
plt.legend()
plt.title('Dice coefficient: Valid')
plt.savefig(savePath+'validDice.png')
plt.close()

100%|██████████| 69/69 [01:32<00:00,  1.34s/it]



0][100]| LGseg: 0.7953 | 


  0%|          | 0/22 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
100%|██████████| 22/22 [01:12<00:00,  3.28s/it]


Found a better version, old dice loss: 0.0 -> New dice loss: 0.027053780452606712. Saving checkpoint...
Epoch: 1/100 | Train Loss: 1.120 | Valid Loss: 0.795

Dice | Train  | BG 0.916 | Nodule 0.007 |
 Valid | BG: 0.990 | Nodule 0.027 |

Time: 2m 45s
Saving losses


  0%|          | 0/69 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
100%|██████████| 69/69 [00:30<00:00,  2.29it/s]


...

Epoch: 99/100 | Train Loss: 0.022 | Valid Loss: 0.013

Dice | Train  | BG 1.000 | Nodule 0.980 |
 Valid | BG: 0.999 | Nodule 0.612 |

Time: 0m 47s
Saving losses


  0%|          | 0/69 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
100%|██████████| 69/69 [00:35<00:00,  1.95it/s]



99][100]| LGseg: 0.0379 | 


  0%|          | 0/22 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
100%|██████████| 22/22 [00:12<00:00,  1.79it/s]


Epoch: 100/100 | Train Loss: 0.024 | Valid Loss: 0.038

Dice | Train  | BG 1.000 | Nodule 0.980 |
 Valid | BG: 0.999 | Nodule 0.619 |

Time: 0m 48s
Saving losses
Training completed in 82m 28s


In [ ]:
!mv /content/dataset/subset0/subset0/* /content/dataset/subset0/
!rm -r /content/dataset/subset0/subset0

mv: cannot stat '/content/dataset/subset0/subset0/*': No such file or directory
rm: cannot remove '/content/dataset/subset0/subset0': No such file or directory


In [ ]:
BASE="/content/LUNA16/Lung-nodule-detection-LUNA-16/dataset"

!mv $BASE/subset0/subset0/* $BASE/subset0/
!rm -r $BASE/subset0/subset0

!mv $BASE/subset1/subset1/* $BASE/subset1/
!rm -r $BASE/subset1/subset1


mv: cannot stat '/content/LUNA16/Lung-nodule-detection-LUNA-16/dataset/subset0/subset0/*': No such file or directory
rm: cannot remove '/content/LUNA16/Lung-nodule-detection-LUNA-16/dataset/subset0/subset0': No such file or directory
mv: cannot stat '/content/LUNA16/Lung-nodule-detection-LUNA-16/dataset/subset1/subset1/*': No such file or directory
rm: cannot remove '/content/LUNA16/Lung-nodule-detection-LUNA-16/dataset/subset1/subset1': No such file or directory


In [ ]:
!ls /content
!ls /content/LUNA16
!ls /content/LUNA16/Lung-nodule-detection-LUNA-16
!ls /content/LUNA16/Lung-nodule-detection-LUNA-16/dataset


drive  sample_data
ls: cannot access '/content/LUNA16': No such file or directory
ls: cannot access '/content/LUNA16/Lung-nodule-detection-LUNA-16': No such file or directory
ls: cannot access '/content/LUNA16/Lung-nodule-detection-LUNA-16/dataset': No such file or directory


In [ ]:
!pwd
!ls /content/LUNA16/Lung-nodule-detection-LUNA-16/dataset

/content/drive/MyDrive/LUNA16/Lung-nodule-detection-LUNA-16
ls: cannot access '/content/LUNA16/Lung-nodule-detection-LUNA-16/dataset': No such file or directory


In [ ]:
!pwd
!ls

/content/drive/MyDrive/LUNA16/Lung-nodule-detection-LUNA-16
data  data_prep  dataset  LICENSE  Plots  README.md  train_codes


In [ ]:
BASE="/content/drive/MyDrive/LUNA16/Lung-nodule-detection-LUNA-16/dataset"

!ls "$BASE"

!mv "$BASE/subset0/subset0/"* "$BASE/subset0/"
!rm -r "$BASE/subset0/subset0"

!mv "$BASE/subset1/subset1/"* "$BASE/subset1/"
!rm -r "$BASE/subset1/subset1"


annotations.csv  subset1  subset3  subset5  subset7  subset9
subset0		 subset2  subset4  subset6  subset8


In [ ]:
import os, sys
sys.path.append(os.path.join(os.getcwd(), 'train_codes'))  # thêm folder train_codes vào path

from LUNA_loader import lunaLoader


ModuleNotFoundError: No module named 'LUNA_loader'

In [ ]:
from LUNA_loader import lunaLoader


ModuleNotFoundError: No module named 'LUNA_loader'

In [ ]:
# =========================
#  TEST EVALUATION SECTION
# =========================

# 1) Tạo test set (đổi 'test' -> 'val' nếu loader của bạn chưa có split test riêng)
testDset = lunaLoader(is_transform=True, split='test', img_size=256)
testDataLoader = data.DataLoader(testDset,
                                 batch_size=16,
                                 shuffle=False,
                                 num_workers=4,
                                 pin_memory=True)

# 2) Load lại model tốt nhất đã lưu ở trên
net.load_state_dict(torch.load(savePath + 'sumnet_best.pt'))
net.eval()

from sklearn.metrics import confusion_matrix

dice_list = []
iou_list  = []
sens_list = []
spec_list = []

with torch.no_grad():
    for imgs, mask in tqdm.tqdm(testDataLoader):
        if use_gpu:
            inputs = imgs.cuda()
            labels = mask.cuda()
        else:
            inputs = imgs
            labels = mask

        # forward
        logits = net(inputs)                    # [B, 2, H, W]
        probs  = F.softmax(logits, dim=1)
        preds  = torch.argmax(probs, dim=1)     # [B, H, W], giá trị 0/1

        # tính metric cho từng ảnh trong batch
        for b in range(preds.size(0)):
            y_true = labels[b].cpu().numpy().astype(np.int32)
            y_pred = preds[b].cpu().numpy().astype(np.int32)

            c = confusion_matrix(y_true.ravel(),
                                 y_pred.ravel(),
                                 labels=[0, 1])
            TN, FP, FN, TP = c.ravel()

            dice = (2.0 * TP) / (2.0 * TP + FP + FN + 1e-6)
            iou  = TP / (TP + FP + FN + 1e-6)
            sens = TP / (TP + FN + 1e-6)
            spec = TN / (TN + FP + 1e-6)

            dice_list.append(dice)
            iou_list.append(iou)
            sens_list.append(sens)
            spec_list.append(spec)

# 3) Tính mean ± std để điền vào Table 1
dice_mean = np.mean(dice_list); dice_std = np.std(dice_list)
iou_mean  = np.mean(iou_list);  iou_std  = np.std(iou_list)
sens_mean = np.mean(sens_list)
spec_mean = np.mean(spec_list)

print("=== Test metrics (per-slice) ===")
print(f"Dice mean ± std: {dice_mean:.4f} ± {dice_std:.4f}")
print(f"IoU  mean ± std: {iou_mean:.4f} ± {iou_std:.4f}")
print(f"Sensitivity    : {sens_mean:.4f}")
print(f"Specificity    : {spec_mean:.4f}")


NameError: name 'lunaLoader' is not defined